# Step 4: Classic ML baselines (random split)
Four classic models on the averaged features. **Development stage:** a random 80/20 split puts neighbouring seconds of the same conversation in training and test, so these scores are too optimistic (fixed from step 14 on).
*Same results as `src/04_baselines.py`.*

**In short** (what this notebook shows)

- **What:** Classic machine learning on a random 80/20 split: linear regression, random forest, gradient boosting, XGBoost.
- **Needs the NoXi data?** Yes. Random split: the same person is in training and test, so the scores are optimistic (development step).
- **Result / use:** Linear regression 0.583, random forest 0.875, gradient boosting 0.884, XGBoost 0.888.

### Dataset
**NoXi** (NOvice eXpert Interaction) - multimodal recordings of expert-novice conversations with continuous engagement labels.
- Dataset page (MultiMediate): https://multimediate.perceptualui.org/datasets/Dataset_NoXi/ (if it is offline, use the links below)
- Paper: A. Cafaro et al., "The NoXi Database: Multimodal Recordings of Mediated Novice-Expert Interactions", ACM ICMI 2017: https://eprints.nottingham.ac.uk/45827 (DOI https://doi.org/10.1145/3136755.3136780)
- Extension NoXi+J (2024): https://arxiv.org/abs/2409.13726
- Data access: University of Augsburg, Human-Centered AI lab - noxi@hcai.eu
- Used here: **24 conversations** (English 026, 027, 028, 029, 030, 058, 062, 063; French 001, 007, 009, 011, 013, 014, 020, 021; German 068, 070, 072, 073, 074, 075, 076, 077) from the NoXi train set:
  eGeMAPS voice features, OpenFace 2.0 face features, engagement labels (25 per second) and the pretrained-Transformer embeddings shipped with the corpus.
- Licence: NoXi is available for research under an End-User Licence Agreement (signed by the supervisor). The data is **not included** here and must not be shared;
  request access on the dataset page and place the downloaded zips as described in `01_extract_data`.

Go to the project folder.

In [2]:
import os
if os.getcwd().endswith("notebooks"):
    os.chdir("..")          # work from the project folder

**Imports.**

In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor

**CCC**, the metric: 1 = perfect agreement.

In [4]:
def ccc(y_true, y_pred):
    # Concordance Correlation Coefficient: 1 = perfect agreement
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    covariance = np.mean((y_true - y_true.mean()) * (y_pred - y_pred.mean()))
    return 2 * covariance / (y_true.var() + y_pred.var() + (y_true.mean() - y_pred.mean()) ** 2 + 1e-9)

**Load the data:** all 802 voice and face columns, and the engagement label.

In [5]:
df = pd.read_parquet("data/features/noxi_engagement.parquet")
cols = []
for c in df.columns:
    if c.startswith("voice_") or c.startswith("face_"):
        cols.append(c)
X = df[cols].fillna(0).to_numpy("float32")      # features (missing = 0)
y = df["engagement"].to_numpy("float32")        # labels
print(len(df), "windows,", len(cols), "features")

48486 windows, 802 features


**Random 80/20 split** of the windows.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

**The four models.**

In [7]:
models = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "GradientBoosting": HistGradientBoostingRegressor(max_iter=300, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8,
                            colsample_bytree=0.8, tree_method="hist", random_state=42, n_jobs=-1),
}

**Train each model and score it on the test part.** The scaler learns only from the training part.

In [8]:
print("model              CCC  Pearson    MAE     R2")
for name, model in models.items():
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    model.fit(X_train_scaled, y_train)
    pred = model.predict(X_test_scaled)
    pearson = np.corrcoef(y_test, pred)[0, 1]
    print(f"{name:18s} {ccc(y_test, pred):5.3f} {pearson:8.3f} {mean_absolute_error(y_test, pred):6.3f} {r2_score(y_test, pred):6.3f}")

model              CCC  Pearson    MAE     R2


LinearRegression   0.583    0.593  0.065 -0.014


RandomForest       0.875    0.889  0.047  0.787


GradientBoosting   0.884    0.893  0.048  0.797


XGBoost            0.888    0.897  0.047  0.803
